# Public Transportation Usage Analysis
## Phase 1 — Data Understanding & Data Quality Audit

**Project objective**

Understand the structure and quality of the public transportation dataset, investigate suspicious patterns, create transparent data-quality flags, and prepare a reproducible dataset for Exploratory Data Analysis (EDA).

**Working dataset:** `processed.csv`  
**Initial size:** 500,000 rows × 31 columns  
**Final working size:** 500,000 rows × 33 columns (includes two investigation flags)

> **Dataset context:** This project uses a synthetic, GTFS-inspired dataset. Findings describe patterns in this dataset and should not be presented as verified real-world transport performance.

### Questions explored
1. What is the dataset's structure and are there missing or duplicate records?
2. Are numeric values, timestamps, and binary indicators within expected ranges?
3. Are passenger, occupancy, service-status, and cancellation fields internally consistent?
4. Which potential data-quality issues should be flagged and documented rather than silently corrected?

### Notebook workflow
1. Load and inspect the source data
2. Audit completeness, uniqueness, types, and value ranges
3. Create a typed working copy
4. Investigate passenger-count and delay/status patterns
5. Add quality flags and summarize findings
6. Save the prepared dataset


# Public Transportation Usage Analysis
## Phase 1: Data Understanding & Cleaning

**Objective:** Understand the dataset, inspect data quality,
and prepare reliable data for EDA and Power BI.

**Dataset:** processed.csv


## 1. Load the dataset

Load the source CSV into a Pandas DataFrame. Update `file_path` if the dataset is stored in a different location on your computer.

In [1]:

import pandas as pd
from pathlib import Path

file_path = Path(r"C:\Users\admin\Documents\Projects\Public_Transportation_Performance_Intelligence\Data\raw\public_transportation.csv")

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")


Dataset loaded successfully!


## 2. Initial data inspection

The next cells inspect sample records, dataset dimensions, column names, data types, and non-null counts.

**What to look for:** confirm that the expected trip-level fields are present and understand which columns are numeric, categorical, or date/time related.

In [30]:

# 1. First 5 rows
df.head()


,trip_id,timestamp,route_id,city,route_type,origin_station,dest_station,boarding_count,alighting_count,passenger_count,...,vehicle_capacity,operates_night,is_electric,is_peak_hour,is_weekend,is_holiday,is_school_hours,is_night_service,anomaly_flag,anomaly_type
0,TRIP_RT_LON_000_0000000,2020-01-01 00:00:00,RT_LON_000,London,rail,STN_LON_0066,STN_LON_0009,0,0,0,...,600,1,1,0,0,1,0,1,0,none
1,TRIP_RT_LON_000_0000001,2020-01-01 01:00:00,RT_LON_000,London,rail,STN_LON_0066,STN_LON_0009,0,0,0,...,600,1,1,0,0,1,0,1,0,none
2,TRIP_RT_LON_000_0000002,2020-01-01 02:00:00,RT_LON_000,London,rail,STN_LON_0066,STN_LON_0009,0,0,0,...,600,1,1,0,0,1,0,1,0,none
3,TRIP_RT_LON_000_0000003,2020-01-01 03:00:00,RT_LON_000,London,rail,STN_LON_0066,STN_LON_0009,0,0,0,...,600,1,1,0,0,1,0,1,0,none
4,TRIP_RT_LON_000_0000004,2020-01-01 04:00:00,RT_LON_000,London,rail,STN_LON_0066,STN_LON_0009,0,0,0,...,600,1,1,0,0,1,0,1,0,none


In [31]:

# 2. Number of rows and columns
df.shape


(500000, 31)

In [32]:

# 3. Column names
df.columns.tolist()


['trip_id',
 'timestamp',
 'route_id',
 'city',
 'route_type',
 'origin_station',
 'dest_station',
 'boarding_count',
 'alighting_count',
 'passenger_count',
 'occupancy_rate',
 'delay_minutes',
 'actual_travel_min',
 'scheduled_travel_min',
 'service_status',
 'cancellation_flag',
 'fare_type',
 'weather_condition',
 'temperature_c',
 'route_length_km',
 'headway_min',
 'vehicle_capacity',
 'operates_night',
 'is_electric',
 'is_peak_hour',
 'is_weekend',
 'is_holiday',
 'is_school_hours',
 'is_night_service',
 'anomaly_flag',
 'anomaly_type']

In [33]:

# 4. Data types and non-null values
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 31 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   trip_id               500000 non-null  str    
 1   timestamp             500000 non-null  str    
 2   route_id              500000 non-null  str    
 3   city                  500000 non-null  str    
 4   route_type            500000 non-null  str    
 5   origin_station        500000 non-null  str    
 6   dest_station          500000 non-null  str    
 7   boarding_count        500000 non-null  int64  
 8   alighting_count       500000 non-null  int64  
 9   passenger_count       500000 non-null  int64  
 10  occupancy_rate        500000 non-null  float64
 11  delay_minutes         500000 non-null  int64  
 12  actual_travel_min     500000 non-null  float64
 13  scheduled_travel_min  500000 non-null  float64
 14  service_status        500000 non-null  str    
 15  cancellatio

## 3. Completeness and duplicate checks

Missing values and duplicate rows can affect later calculations. We check both at the full-row level and through the trip identifier.

In [35]:

# Count missing values in each column
missing_values = df.isnull().sum()

# Show only columns that have missing values
missing_values[missing_values > 0].sort_values(ascending=False)


Series([], dtype: int64)

In [36]:

missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().mean() * 100).round(2)
})

missing_summary = missing_summary[
    missing_summary["Missing Count"] > 0
].sort_values("Missing Percentage", ascending=False)

missing_summary


,Missing Count,Missing Percentage


In [38]:

# Count completely duplicated rows
duplicate_count = df.duplicated().sum()

print("Total duplicate rows:", duplicate_count)


Total duplicate rows: 0


In [39]:

print("Total rows:", len(df))
print("Unique trip IDs:", df["trip_id"].nunique())
print("Missing trip IDs:", df["trip_id"].isnull().sum())


Total rows: 500000
Unique trip IDs: 500000
Missing trip IDs: 0


## 4. Data types and timestamp inspection

The timestamp is initially read from CSV as text. We inspect it before converting it to a datetime type in the cleaned working copy.

In [42]:

# Check data types
df.dtypes


trip_id                     str
timestamp                   str
route_id                    str
city                        str
route_type                  str
origin_station              str
dest_station                str
boarding_count            int64
alighting_count           int64
passenger_count           int64
occupancy_rate          float64
delay_minutes             int64
actual_travel_min       float64
scheduled_travel_min    float64
service_status              str
cancellation_flag         int64
fare_type                   str
weather_condition           str
temperature_c           float64
route_length_km         float64
headway_min               int64
vehicle_capacity          int64
operates_night            int64
is_electric               int64
is_peak_hour              int64
is_weekend                int64
is_holiday                int64
is_school_hours           int64
is_night_service          int64
anomaly_flag              int64
anomaly_type                str
dtype: o

In [43]:

# Check timestamp values
print("Timestamp data type:", df["timestamp"].dtype)
print("Missing timestamps:", df["timestamp"].isnull().sum())

# Preview timestamp values
df["timestamp"].head(10)


Timestamp data type: str
Missing timestamps: 0


0    2020-01-01 00:00:00
1    2020-01-01 01:00:00
2    2020-01-01 02:00:00
3    2020-01-01 03:00:00
4    2020-01-01 04:00:00
5    2020-01-01 05:00:00
6    2020-01-01 06:00:00
7    2020-01-01 07:00:00
8    2020-01-01 08:00:00
9    2020-01-01 09:00:00
Name: timestamp, dtype: str

## 5. Numeric summaries and range validation

Use descriptive statistics and rule-based checks to identify impossible or suspicious values. A value being zero is not automatically an error; its meaning depends on the field and dataset design.

In [45]:

numeric_columns = [
    "boarding_count",
    "alighting_count",
    "passenger_count",
    "occupancy_rate",
    "delay_minutes",
    "actual_travel_min",
    "scheduled_travel_min",
    "temperature_c",
    "route_length_km",
    "headway_min",
    "vehicle_capacity"
]

df[numeric_columns].describe().T


,count,mean,std,min,25%,50%,75%,max
boarding_count,500000.0,1.368886,2.523518,0.00,0.0,0.000,2.000,54.00
alighting_count,500000.0,1.317672,2.368527,0.00,0.0,0.000,2.000,22.00
passenger_count,500000.0,78.915102,154.153505,0.00,0.0,7.000,66.000,800.00
occupancy_rate,500000.0,0.178834,0.274171,0.00,0.0,0.014,0.295,1.00
delay_minutes,500000.0,0.652154,3.145602,0.00,0.0,0.000,0.000,86.00
actual_travel_min,500000.0,68.490851,32.578661,1.90,40.8,72.000,97.000,153.20
scheduled_travel_min,500000.0,68.167808,32.422257,10.60,41.2,72.200,97.400,119.00
temperature_c,500000.0,14.994821,7.683029,-6.97,8.4,14.970,21.600,37.81
route_length_km,500000.0,34.083904,16.211128,5.30,20.6,36.100,48.700,59.50
headway_min,500000.0,19.067360,16.970422,2.00,6.0,11.000,30.000,59.00


In [46]:

count_columns = [
    "boarding_count",
    "alighting_count",
    "passenger_count",
    "actual_travel_min",
    "scheduled_travel_min",
    "route_length_km",
    "headway_min",
    "vehicle_capacity"
]

negative_summary = (df[count_columns] < 0).sum()

negative_summary[negative_summary > 0]


Series([], dtype: int64)

In [47]:

# Occupancy rate range
print("Occupancy below 0:", (df["occupancy_rate"] < 0).sum())
print("Occupancy above 1:", (df["occupancy_rate"] > 1).sum())

# Check expected binary flags
flag_columns = [
    "cancellation_flag",
    "operates_night",
    "is_electric",
    "is_peak_hour",
    "is_weekend",
    "is_holiday",
    "is_school_hours",
    "is_night_service",
    "anomaly_flag"
]

for col in flag_columns:
    print(col, "->", df[col].value_counts(dropna=False).to_dict())


Occupancy below 0: 0
Occupancy above 1: 0
cancellation_flag -> {0: 479930, 1: 20070}
operates_night -> {0: 386120, 1: 113880}
is_electric -> {1: 315360, 0: 184640}
is_peak_hour -> {0: 354168, 1: 145832}
is_weekend -> {0: 357536, 1: 142464}
is_holiday -> {0: 490400, 1: 9600}
is_school_hours -> {0: 383112, 1: 116888}
is_night_service -> {0: 354163, 1: 145837}
anomaly_flag -> {0: 496972, 1: 3028}


## 6. Passenger counts and occupancy investigation

Passenger-related counts can legitimately contain zeros. We compare boarding, alighting, passenger count, occupancy rate, and vehicle capacity to understand their relationships.

In [48]:

# Check how many records have zero passenger-related counts
passenger_columns = [
    "boarding_count",
    "alighting_count",
    "passenger_count"
]

for col in passenger_columns:
    zero_count = (df[col] == 0).sum()
    print(
        f"{col}: {zero_count:,} zero values "
        f"({zero_count / len(df) * 100:.2f}%)"
    )


boarding_count: 275,867 zero values (55.17%)
alighting_count: 275,867 zero values (55.17%)
passenger_count: 211,359 zero values (42.27%)


In [49]:

# Inspect records where passenger_count differs from boarding_count
df.loc[
    df["passenger_count"] != df["boarding_count"],
    [
        "trip_id",
        "boarding_count",
        "alighting_count",
        "passenger_count",
        "occupancy_rate",
        "vehicle_capacity"
    ]
].head(10)


,trip_id,boarding_count,alighting_count,passenger_count,occupancy_rate,vehicle_capacity
6,TRIP_RT_LON_000_0000006,1,1,0,0.0,600
7,TRIP_RT_LON_000_0000007,2,2,0,0.0,600
8,TRIP_RT_LON_000_0000008,3,3,0,0.0,600
9,TRIP_RT_LON_000_0000009,2,2,0,0.0,600
14,TRIP_RT_LON_000_0000014,1,1,0,0.0,600
16,TRIP_RT_LON_000_0000016,2,2,0,0.0,600
17,TRIP_RT_LON_000_0000017,2,2,0,0.0,600
18,TRIP_RT_LON_000_0000018,3,3,0,0.0,600
19,TRIP_RT_LON_000_0000019,1,1,0,0.0,600
21,TRIP_RT_LON_000_0000021,1,1,0,0.0,600


In [50]:

# Compare occupancy_rate with passenger_count / vehicle_capacity
df["calculated_occupancy"] = (
    df["passenger_count"] / df["vehicle_capacity"]
)

df[
    [
        "passenger_count",
        "vehicle_capacity",
        "occupancy_rate",
        "calculated_occupancy"
    ]
].head(10)


,passenger_count,vehicle_capacity,occupancy_rate,calculated_occupancy
0,0,600,0.0,0.0
1,0,600,0.0,0.0
2,0,600,0.0,0.0
3,0,600,0.0,0.0
4,0,600,0.0,0.0
5,0,600,0.0,0.0
6,0,600,0.0,0.0
7,0,600,0.0,0.0
8,0,600,0.0,0.0
9,0,600,0.0,0.0


In [51]:
# Compare reported and calculated occupancy
valid_capacity = df["vehicle_capacity"] > 0

occupancy_check = (
    df.loc[valid_capacity, "occupancy_rate"]
    - (
        df.loc[valid_capacity, "passenger_count"]
        / df.loc[valid_capacity, "vehicle_capacity"]
    )
).abs()

print("Rows with valid capacity:", valid_capacity.sum())
print("Exact occupancy matches:", (occupancy_check < 0.001).sum())
print("Occupancy mismatches:", (occupancy_check >= 0.001).sum())
print("Rows with zero capacity:", (df["vehicle_capacity"] == 0).sum())

Rows with valid capacity: 500000
Exact occupancy matches: 500000
Occupancy mismatches: 0
Rows with zero capacity: 0


In [52]:
condition = (
    (df["boarding_count"] > 0)
    & (df["passenger_count"] == 0)
)

print("Boarding positive but passenger count zero:", condition.sum())

df.loc[
    condition,
    [
        "trip_id",
        "city",
        "route_type",
        "service_status",
        "boarding_count",
        "alighting_count",
        "passenger_count",
        "occupancy_rate",
        "vehicle_capacity"
    ]
].head(15)

Boarding positive but passenger count zero: 46333


,trip_id,city,route_type,service_status,boarding_count,alighting_count,passenger_count,occupancy_rate,vehicle_capacity
6,TRIP_RT_LON_000_0000006,London,rail,on_time,1,1,0,0.0,600
7,TRIP_RT_LON_000_0000007,London,rail,diverted,2,2,0,0.0,600
8,TRIP_RT_LON_000_0000008,London,rail,on_time,3,3,0,0.0,600
9,TRIP_RT_LON_000_0000009,London,rail,minor_delay,2,2,0,0.0,600
14,TRIP_RT_LON_000_0000014,London,rail,on_time,1,1,0,0.0,600
16,TRIP_RT_LON_000_0000016,London,rail,on_time,2,2,0,0.0,600
17,TRIP_RT_LON_000_0000017,London,rail,minor_delay,2,2,0,0.0,600
18,TRIP_RT_LON_000_0000018,London,rail,major_delay,3,3,0,0.0,600
19,TRIP_RT_LON_000_0000019,London,rail,on_time,1,1,0,0.0,600
21,TRIP_RT_LON_000_0000021,London,rail,on_time,1,1,0,0.0,600


In [53]:
pd.crosstab(
    df["service_status"],
    df["boarding_count"].eq(0),
    normalize="index"
).round(3) * 100

boarding_count,False,True
service_status,,
cancelled,44.3,55.7
diverted,46.0,54.0
major_delay,45.0,55.0
minor_delay,45.0,55.0
on_time,44.8,55.2


## 7. Categorical flags and service consistency

Check binary indicator values and compare related fields such as `service_status`, `cancellation_flag`, `anomaly_flag`, and `anomaly_type`.

In [54]:

flag_columns = [
    "cancellation_flag",
    "operates_night",
    "is_electric",
    "is_peak_hour",
    "is_weekend",
    "is_holiday",
    "is_school_hours",
    "is_night_service",
    "anomaly_flag"
]

for col in flag_columns:
    values = sorted(df[col].dropna().unique().tolist())
    print(f"{col}: {values}")


cancellation_flag: [0, 1]
operates_night: [0, 1]
is_electric: [0, 1]
is_peak_hour: [0, 1]
is_weekend: [0, 1]
is_holiday: [0, 1]
is_school_hours: [0, 1]
is_night_service: [0, 1]
anomaly_flag: [0, 1]


In [55]:

pd.crosstab(
    df["service_status"],
    df["cancellation_flag"],
    margins=True
)


cancellation_flag,0,1,All
service_status,,,
cancelled,0,20070,20070
diverted,10016,0,10016
major_delay,34876,0,34876
minor_delay,75112,0,75112
on_time,359926,0,359926
All,479930,20070,500000


In [56]:

pd.crosstab(
    df["anomaly_flag"],
    df["anomaly_type"],
    dropna=False,
    margins=True
)


anomaly_type,cascading_delay,demand_spike,none,service_drop,All
anomaly_flag,,,,,
0,0,0,496972,0,496972
1,85,1931,0,1012,3028
All,85,1931,496972,1012,500000


## 8. Timestamp and identifier validation

Validate timestamp parsing, date coverage, duplicate trip IDs, and duplicate trip/timestamp combinations.

In [57]:

timestamp_parsed = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

print("Original missing timestamps:", df["timestamp"].isna().sum())
print("Invalid timestamps:", timestamp_parsed.isna().sum())

print("Earliest timestamp:", timestamp_parsed.min())
print("Latest timestamp:", timestamp_parsed.max())


Original missing timestamps: 0
Invalid timestamps: 0
Earliest timestamp: 2020-01-01 00:00:00
Latest timestamp: 2020-12-30 23:00:00


In [58]:

print("Duplicate trip IDs:", df["trip_id"].duplicated().sum())

print(
    "Duplicate trip ID + timestamp pairs:",
    df.duplicated(subset=["trip_id", "timestamp"]).sum()
)


Duplicate trip IDs: 0
Duplicate trip ID + timestamp pairs: 0


In [59]:

print(
    "Records outside 2020:",
    (
        timestamp_parsed.notna()
        & (timestamp_parsed.dt.year != 2020)
    ).sum()
)

print(
    "Rows with missing parsed timestamp:",
    timestamp_parsed.isna().sum()
)


Records outside 2020: 0
Rows with missing parsed timestamp: 0


## 9. Consolidated source-data quality checks

Run a final set of checks for missingness, numeric ranges, duplicate column names, and memory usage before creating the working copy.

In [60]:

missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
})

missing_summary.sort_values(
    "missing_count",
    ascending=False
).head(15)


,missing_count,missing_percent
trip_id,0,0.0
timestamp,0,0.0
route_id,0,0.0
city,0,0.0
route_type,0,0.0
origin_station,0,0.0
dest_station,0,0.0
boarding_count,0,0.0
alighting_count,0,0.0
passenger_count,0,0.0


In [61]:

range_checks = {
    "boarding_count": df["boarding_count"] < 0,
    "alighting_count": df["alighting_count"] < 0,
    "passenger_count": df["passenger_count"] < 0,
    "occupancy_rate": (
        (df["occupancy_rate"] < 0)
        | (df["occupancy_rate"] > 1)
    ),
    "vehicle_capacity": df["vehicle_capacity"] <= 0,
    "actual_travel_min": df["actual_travel_min"] <= 0,
    "scheduled_travel_min": df["scheduled_travel_min"] <= 0,
    "route_length_km": df["route_length_km"] <= 0,
    "headway_min": df["headway_min"] <= 0
}

for col, condition in range_checks.items():
    print(f"{col}: {condition.sum()} suspicious rows")


boarding_count: 0 suspicious rows
alighting_count: 0 suspicious rows
passenger_count: 0 suspicious rows
occupancy_rate: 0 suspicious rows
vehicle_capacity: 0 suspicious rows
actual_travel_min: 0 suspicious rows
scheduled_travel_min: 0 suspicious rows
route_length_km: 0 suspicious rows
headway_min: 0 suspicious rows


In [62]:

print("Dataset shape:", df.shape)
print("Unique columns:", df.columns.nunique())
print("Duplicate column names:", df.columns[df.columns.duplicated()].tolist())
print("Memory usage (MB):", round(df.memory_usage(deep=True).sum() / 1024**2, 2))


Dataset shape: (500000, 32)
Unique columns: 32
Duplicate column names: []
Memory usage (MB): 175.74


## 10. Create the cleaned working copy

Keep the original `df` unchanged. Create `df_clean` as a separate working copy and convert `timestamp` to datetime.

The temporary `calculated_occupancy` column was used only for investigation and is removed from the working copy in the next step.

In [64]:

# Create a separate working copy
df_clean = df.copy()

# Convert timestamp to datetime
df_clean["timestamp"] = pd.to_datetime(
    df_clean["timestamp"],
    errors="raise"
)

# Verify the change
print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
print("Timestamp dtype:", df_clean["timestamp"].dtype)
print("Missing values:", df_clean.isna().sum().sum())


Original shape: (500000, 32)
Cleaned shape: (500000, 32)
Timestamp dtype: datetime64[us]
Missing values: 0


In [65]:

print("Original columns:", len(df.columns))
print("Cleaned columns:", len(df_clean.columns))

print("\nColumns in df:")
print(df.columns.tolist())

print("\nColumns in df_clean:")
print(df_clean.columns.tolist())


Original columns: 32
Cleaned columns: 32

Columns in df:
['trip_id', 'timestamp', 'route_id', 'city', 'route_type', 'origin_station', 'dest_station', 'boarding_count', 'alighting_count', 'passenger_count', 'occupancy_rate', 'delay_minutes', 'actual_travel_min', 'scheduled_travel_min', 'service_status', 'cancellation_flag', 'fare_type', 'weather_condition', 'temperature_c', 'route_length_km', 'headway_min', 'vehicle_capacity', 'operates_night', 'is_electric', 'is_peak_hour', 'is_weekend', 'is_holiday', 'is_school_hours', 'is_night_service', 'anomaly_flag', 'anomaly_type', 'calculated_occupancy']

Columns in df_clean:
['trip_id', 'timestamp', 'route_id', 'city', 'route_type', 'origin_station', 'dest_station', 'boarding_count', 'alighting_count', 'passenger_count', 'occupancy_rate', 'delay_minutes', 'actual_travel_min', 'scheduled_travel_min', 'service_status', 'cancellation_flag', 'fare_type', 'weather_condition', 'temperature_c', 'route_length_km', 'headway_min', 'vehicle_capacity', 'op

In [66]:

# Remove the temporary investigation column from the cleaned copy
df_clean = df_clean.drop(columns=["calculated_occupancy"])

# Verify
print("Cleaned dataset shape:", df_clean.shape)
print("Temporary column exists:", "calculated_occupancy" in df_clean.columns)
print("Missing values:", df_clean.isna().sum().sum())


Cleaned dataset shape: (500000, 31)
Temporary column exists: False
Missing values: 0


In [67]:

df_clean.info()


<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 31 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   trip_id               500000 non-null  str           
 1   timestamp             500000 non-null  datetime64[us]
 2   route_id              500000 non-null  str           
 3   city                  500000 non-null  str           
 4   route_type            500000 non-null  str           
 5   origin_station        500000 non-null  str           
 6   dest_station          500000 non-null  str           
 7   boarding_count        500000 non-null  int64         
 8   alighting_count       500000 non-null  int64         
 9   passenger_count       500000 non-null  int64         
 10  occupancy_rate        500000 non-null  float64       
 11  delay_minutes         500000 non-null  int64         
 12  actual_travel_min     500000 non-null  float64       
 13  scheduled_

## 11. Investigate boarding/passenger mismatch

**Investigation rule:** `boarding_count > 0` while `passenger_count == 0`.

This rule identifies records for review; it does not prove which field is incorrect. We inspect examples, compare cities and route types, and calculate rates using each group's total records as the denominator.

In [68]:
# Records where boarding is positive but passengers are zero
suspicious_records = df_clean[
    (df_clean["boarding_count"] > 0) &
    (df_clean["passenger_count"] == 0)
]

print("Suspicious records:", len(suspicious_records))

display(
    suspicious_records[
        [
            "city",
            "route_type",
            "boarding_count",
            "alighting_count",
            "passenger_count",
            "occupancy_rate",
            "service_status"
        ]
    ].head(10)
)

Suspicious records: 46333


,city,route_type,boarding_count,alighting_count,passenger_count,occupancy_rate,service_status
6,London,rail,1,1,0,0.0,on_time
7,London,rail,2,2,0,0.0,diverted
8,London,rail,3,3,0,0.0,on_time
9,London,rail,2,2,0,0.0,minor_delay
14,London,rail,1,1,0,0.0,on_time
16,London,rail,2,2,0,0.0,on_time
17,London,rail,2,2,0,0.0,minor_delay
18,London,rail,3,3,0,0.0,major_delay
19,London,rail,1,1,0,0.0,on_time
21,London,rail,1,1,0,0.0,on_time


In [69]:
# Count suspicious records by city
city_check = (
    suspicious_records["city"]
    .value_counts()
    .rename_axis("city")
    .reset_index(name="suspicious_records")
)

display(city_check)

,city,suspicious_records
0,London,18352
1,Paris,17589
2,Berlin,9639
3,Tokyo,525
4,New_York,228


In [70]:
route_check = (
    suspicious_records["route_type"]
    .value_counts()
    .rename_axis("route_type")
    .reset_index(name="suspicious_records")
)

display(route_check)

,route_type,suspicious_records
0,rail,15019
1,bus,13718
2,tram,12381
3,metro,5215


In [71]:
display(
    suspicious_records[
        ["boarding_count", "alighting_count"]
    ].describe()
)

,boarding_count,alighting_count
count,46333.000000,46333.000000
mean,1.279671,1.268858
std,0.675480,0.639161
min,1.000000,1.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,1.000000,1.000000
max,18.000000,17.000000


In [72]:
# City-wise data quality
city_total = df_clean["city"].value_counts()

city_suspicious = suspicious_records["city"].value_counts()

city_quality = pd.DataFrame({
    "total_records": city_total,
    "suspicious_records": city_suspicious
}).fillna(0)

city_quality["suspicious_records"] = (
    city_quality["suspicious_records"].astype(int)
)

city_quality["suspicious_percentage"] = (
    city_quality["suspicious_records"]
    / city_quality["total_records"] * 100
).round(2)

display(
    city_quality.sort_values(
        "suspicious_percentage", ascending=False
    )
)

,total_records,suspicious_records,suspicious_percentage
city,,,
London,96360,18352,19.05
Paris,123320,17589,14.26
Berlin,87600,9639,11.00
Tokyo,113880,525,0.46
New_York,78840,228,0.29


In [73]:
route_total = df_clean["route_type"].value_counts()

route_suspicious = suspicious_records["route_type"].value_counts()

route_quality = pd.DataFrame({
    "total_records": route_total,
    "suspicious_records": route_suspicious
}).fillna(0)

route_quality["suspicious_records"] = (
    route_quality["suspicious_records"].astype(int)
)

route_quality["suspicious_percentage"] = (
    route_quality["suspicious_records"]
    / route_quality["total_records"] * 100
).round(2)

display(
    route_quality.sort_values(
        "suspicious_percentage", ascending=False
    )
)

,total_records,suspicious_records,suspicious_percentage
route_type,,,
tram,70080,12381,17.67
bus,123320,13718,11.12
rail,166440,15019,9.02
metro,140160,5215,3.72


### Finding 1 — Boarding/passenger potential mismatch

The investigation identified **46,333 records (9.27%)** matching the rule.

Observed group rates:
- London: **19.05%**
- Paris: **14.26%**
- Berlin: **11.00%**
- Tokyo: **0.46%**
- New York: **0.29%**

By route type:
- Tram: **17.67%**
- Bus: **11.12%**
- Rail: **9.02%**
- Metro: **3.72%**

**Interpretation:** The pattern varies substantially by city and route type. These are observed associations in a synthetic dataset; they do not establish the cause of the mismatch.

**Decision:** Preserve the original values and add a flag so the records can be filtered or investigated later.

In [74]:
df_clean["boarding_passenger_mismatch"] = (
    (df_clean["boarding_count"] > 0) &
    (df_clean["passenger_count"] == 0)
).astype("int8")

print(
    df_clean["boarding_passenger_mismatch"]
    .value_counts()
)

print(
    "Mismatch percentage:",
    round(
        df_clean["boarding_passenger_mismatch"].mean() * 100,
        2
    ),
    "%"
)

boarding_passenger_mismatch
0    453667
1     46333
Name: count, dtype: int64
Mismatch percentage: 9.27 %


## 12. Delay distribution and service-status investigation

First inspect delay summaries and the share of zero/negative delays by status. Then inspect the full delay distribution rather than relying on the mean alone.

In [75]:
delay_summary = (
    df_clean.groupby("service_status")["delay_minutes"]
    .agg(["count", "min", "mean", "median", "max"])
    .round(2)
)

display(delay_summary)

,count,min,mean,median,max
service_status,,,,,
cancelled,20070,0,0.63,0.0,71
diverted,10016,0,0.65,0.0,71
major_delay,34876,0,0.67,0.0,73
minor_delay,75112,0,0.64,0.0,65
on_time,359926,0,0.65,0.0,86


In [76]:
delay_check = (
    df_clean.assign(
        zero_or_negative_delay=df_clean["delay_minutes"] <= 0
    )
    .groupby("service_status")
    .agg(
        total_records=("delay_minutes", "size"),
        zero_or_negative=("zero_or_negative_delay", "sum")
    )
)

delay_check["percentage"] = (
    delay_check["zero_or_negative"]
    / delay_check["total_records"] * 100
).round(2)

display(delay_check)

,total_records,zero_or_negative,percentage
service_status,,,
cancelled,20070,18559,92.47
diverted,10016,9240,92.25
major_delay,34876,32149,92.18
minor_delay,75112,69360,92.34
on_time,359926,332128,92.28


In [77]:
print("Negative delays:", (df_clean["delay_minutes"] < 0).sum())
print("Zero delays:", (df_clean["delay_minutes"] == 0).sum())
print("Positive delays:", (df_clean["delay_minutes"] > 0).sum())

display(
    df_clean["delay_minutes"]
    .value_counts()
    .sort_index()
    .head(20)
)

Negative delays: 0
Zero delays: 461436
Positive delays: 38564


delay_minutes
0     461436
1       4552
2       4091
3       3523
4       3087
5       2759
6       2389
7       2137
8       1884
9       1624
10      1506
11      1345
12      1145
13       959
14       891
15       767
16       688
17       619
18       545
19       511
Name: count, dtype: int64

In [78]:
delay_distribution = pd.crosstab(
    df_clean["service_status"],
    pd.cut(
        df_clean["delay_minutes"],
        bins=[-float("inf"), -1, 0, 5, 15, 30, float("inf")],
        labels=[
            "Below 0",
            "Exactly 0",
            "1–5 min",
            "6–15 min",
            "16–30 min",
            "Above 30 min"
        ]
    )
)

display(delay_distribution)

delay_minutes,Exactly 0,1–5 min,6–15 min,16–30 min,Above 30 min
service_status,,,,,
cancelled,18559,738,541,190,42
diverted,9240,368,305,83,20
major_delay,32149,1221,1072,372,62
minor_delay,69360,2758,2131,731,132
on_time,332128,12927,10598,3664,609


### Finding 2 — Potential service-status/delay mismatch

The investigation rule flags:
- `on_time` records with `delay_minutes > 0`
- `minor_delay` or `major_delay` records with `delay_minutes == 0`

The resulting flag identifies **129,307 records (25.86%)**.

**Interpretation:** This is a potential semantic inconsistency under the rule defined for this analysis. It is not a confirmed error because the dataset's status-generation rules have not been independently verified.

**Decision:** Preserve the source values and retain a separate investigation flag. `cancelled` and `diverted` records are excluded from this particular rule because their delay semantics require dataset-specific documentation.

In [79]:
df_clean["status_delay_mismatch"] = (
    (
        (df_clean["service_status"] == "on_time") &
        (df_clean["delay_minutes"] > 0)
    ) |
    (
        df_clean["service_status"].isin(
            ["minor_delay", "major_delay"]
        ) &
        (df_clean["delay_minutes"] == 0)
    )
).astype("int8")

print(
    df_clean["status_delay_mismatch"].value_counts()
)

print(
    "Potential mismatch percentage:",
    round(
        df_clean["status_delay_mismatch"].mean() * 100,
        2
    ),
    "%"
)

status_delay_mismatch
0    370693
1    129307
Name: count, dtype: int64
Potential mismatch percentage: 25.86 %


In [80]:
quality_summary = pd.DataFrame({
    "flag": [
        "Boarding/passenger mismatch",
        "Status/delay mismatch"
    ],
    "flagged_records": [
        df_clean["boarding_passenger_mismatch"].sum(),
        df_clean["status_delay_mismatch"].sum()
    ]
})

quality_summary["percentage"] = (
    quality_summary["flagged_records"]
    / len(df_clean) * 100
).round(2)

display(quality_summary)

,flag,flagged_records,percentage
0,Boarding/passenger mismatch,46333,9.27
1,Status/delay mismatch,129307,25.86


## 13. Cancellation consistency check

Compare `cancellation_flag` with the expected indicator derived from `service_status == "cancelled"`.

In [81]:
# Expected cancellation flag from service status
expected_cancellation = (
    df_clean["service_status"] == "cancelled"
).astype("int8")

# Compare actual and expected flags
cancellation_mismatch = (
    df_clean["cancellation_flag"] != expected_cancellation
)

print(
    "Cancellation mismatches:",
    cancellation_mismatch.sum()
)

print(
    "Cancellation mismatch percentage:",
    round(cancellation_mismatch.mean() * 100, 2),
    "%"
)

Cancellation mismatches: 0
Cancellation mismatch percentage: 0.0 %


## 14. Final data-quality audit summary

The final summary consolidates the key checks. The two potential mismatch counts are flags for investigation, not automatic corrections.

In [82]:
audit_summary = pd.DataFrame({
    "Check": [
        "Total records",
        "Total columns before adding quality flags",
        "Missing values",
        "Duplicate rows",
        "Duplicate trip IDs",
        "Invalid timestamps",
        "Boarding/passenger potential mismatch",
        "Status/delay potential mismatch",
        "Cancellation flag mismatch"
    ],
    "Result": [
        len(df_clean),
        31,
        int(df_clean.isna().sum().sum()),
        int(df_clean.duplicated().sum()),
        int(df_clean["trip_id"].duplicated().sum()),
        int(df_clean["timestamp"].isna().sum()),
        int(df_clean["boarding_passenger_mismatch"].sum()),
        int(df_clean["status_delay_mismatch"].sum()),
        int(cancellation_mismatch.sum())
    ]
})

display(audit_summary)

,Check,Result
0,Total records,500000
1,Total columns before adding quality flags,31
2,Missing values,0
3,Duplicate rows,0
4,Duplicate trip IDs,0
5,Invalid timestamps,0
6,Boarding/passenger potential mismatch,46333
7,Status/delay potential mismatch,129307
8,Cancellation flag mismatch,0


## 15. Export the prepared dataset

Save the working dataset to the project's `Data/processed` folder. The source CSV remains untouched.

The exported file includes the parsed timestamp and two additional quality flags:
- `boarding_passenger_mismatch`
- `status_delay_mismatch`

**Reproducibility:** Run the notebook from top to bottom after confirming that `file_path` points to your source CSV and that the output folder is writable.

In [86]:
from pathlib import Path

project_folder = Path.home() / "Documents" / "Public_Transportation" / "Data" / "processed"
project_folder.mkdir(parents=True, exist_ok=True)

output_path = project_folder / "public_transportation_cleaned.csv"

df_clean.to_csv(output_path, index=False)

print("Saved successfully:", output_path.resolve())
print("Final shape:", df_clean.shape)
print("File size (MB):", round(output_path.stat().st_size / (1024**2), 2))

Saved successfully: C:\Users\admin\Documents\Public_Transportation\Data\processed\public_transportation_cleaned.csv
Final shape: (500000, 33)
File size (MB): 89.46


## 16. Phase 1 — Key findings and decisions

### Dataset profile
- **500,000 records** and **31 original columns**
- **0 missing values**
- **0 duplicate rows**
- **0 duplicate trip IDs**
- Timestamp range: **2020-01-01 00:00:00 to 2020-12-30 23:00:00**
- **0 invalid timestamps** and **0 records outside 2020**
- Occupancy rate matched `passenger_count / vehicle_capacity` for all 500,000 records in the audit.
- `cancellation_flag` matched the expected flag from `service_status` with **0 mismatches**.

### Potential data-quality patterns
| Investigation flag | Records | Share |
|---|---:|---:|
| Boarding/passenger mismatch | 46,333 | 9.27% |
| Status/delay mismatch | 129,307 | 25.86% |

These flags are intended for transparency and further investigation. No source values were overwritten or deleted based on these patterns.

### Preparation decisions
1. Converted `timestamp` to datetime in `df_clean`.
2. Kept the original `df` unchanged.
3. Removed the temporary `calculated_occupancy` investigation column from `df_clean`.
4. Added two integer quality flags.
5. Exported the prepared dataset separately from the source file.

### Limitations
- The dataset is synthetic; findings should not be generalized to actual transit systems.
- The definitions of `passenger_count` and `service_status` should be confirmed from dataset documentation before attempting any corrections.
- This phase focuses on data understanding and quality. It does not yet establish causal explanations or predictive relationships.

---

## Next phase — Exploratory Data Analysis (EDA)

The prepared dataset can now be used to explore:
- passenger demand by city and route type
- hourly, daily, and monthly usage patterns
- delay and service reliability patterns
- occupancy and vehicle capacity
- anomalies and operational patterns

**Suggested next notebook:** `02_Public_Transportation_EDA.ipynb`